In [ ]:
!pip install gtts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 11.0 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.2.1
    Uninstalling click-8.2.1:
      Successfully uninstalled click-8.2.1


In [ ]:
!pip install transformers torch accelerate

In [ ]:
import torch
import pandas as pd
from transformers import AutoTokenizer, pipeline, MarianMTModel, MarianTokenizer
from datetime import datetime
from gtts import gTTS
import random
import IPython.display as ipd

In [ ]:
# Creating Huggingface
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: read).
The token `qwerty` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `qwerty`


In [ ]:
!hf auth whoami

dikshagarg


# LOAD MODELS

In [ ]:
# Main LLM for excuse generation
MODEL_NAME = "HuggingFaceH4/zephyr-7b-beta"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

excuse_generator = pipeline(
    "text-generation",
    model=MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

...

model-00004-of-00008.safetensors:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

model-00008-of-00008.safetensors:   0%|          | 0.00/816M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Device set to use cuda:0


#   Supportive Text

In [ ]:
def generate_supportive_text(scenario):
    followups = {
        "work": ["I will submit the report by evening.", "I’ll catch up during the weekend."],
        "school": ["I’ll complete the homework tomorrow.", "I’ll ask a friend to share notes."],
        "social": ["Let’s meet another day soon.", "I’ll call you tomorrow."],
        "family": ["I’ll help out later tonight.", "We can do it on Sunday instead."]
    }
    return random.choice(followups.get(scenario, ["I’ll make up for it soon."]))

# translation model


In [ ]:
# Translation model (English -> Hindi)
TRANS_MODEL = "Helsinki-NLP/opus-mt-en-hi"
trans_tokenizer = MarianTokenizer.from_pretrained(TRANS_MODEL)
trans_model = MarianMTModel.from_pretrained(TRANS_MODEL)

def translate_to_hindi(text: str) -> str:
    inputs = trans_tokenizer(text, return_tensors="pt", padding=True)
    translated = trans_model.generate(**inputs)
    return trans_tokenizer.decode(translated[0], skip_special_tokens=True)

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/812k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/306M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/306M [00:00<?, ?B/s]

# Excuse Generation

In [ ]:
HISTORY_FILE = "/content/excuse_history.csv"

def generate_excuse(scenario="work", tone="professional", urgency="medium", language="en"):
    """
    Generate a believable excuse with scenario, tone, urgency, language.
    """

    prompt = (
        f"Generate a short, believable excuse.\n"
        f"Scenario: {scenario}\n"
        f"Tone: {tone}\n"
        f"Urgency: {urgency}\n"
        f"Provide the excuse only, no extra text."
    )

    output = excuse_generator(
        prompt,
        do_sample=True,
        top_k=50,
        max_new_tokens=80,
        pad_token_id=tokenizer.eos_token_id
    )

    excuse = output[0]["generated_text"].split("Excuse:")[-1].strip()

    # Translate if needed
    if language == "hi":
        excuse = translate_to_hindi(excuse)

    # Add supportive "proof-like" details (safe)
    supportive = generate_supportive_text(scenario)

    # Save to history
    log_excuse(scenario, tone, urgency, language, excuse, supportive)

    return excuse, supportive

# History & Favorites


In [ ]:
def log_excuse(scenario, tone, urgency, language, excuse, supportive):
    row = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "scenario": scenario,
        "tone": tone,
        "urgency": urgency,
        "language": language,
        "excuse": excuse,
        "supportive": supportive,
        "clarity_score": clarity_score(excuse),
        "empathy_score": empathy_score(excuse)
    }
    try:
        df = pd.read_csv(HISTORY_FILE)
    except FileNotFoundError:
        df = pd.DataFrame(columns=row.keys())
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df.to_csv(HISTORY_FILE, index=False)

def show_history(n=5):
    try:
        df = pd.read_csv(HISTORY_FILE)
        return df.tail(n)
    except FileNotFoundError:
        return "No history yet."

def save_favorite(index: int):
    df = pd.read_csv(HISTORY_FILE)
    fav = df.iloc[index]
    fav.to_frame().T.to_csv("/content/favorites.csv", mode="a", header=False, index=False)
    return f"Saved to favorites: {fav.excuse}"

def show_favorites():
    try:
        favs = pd.read_csv("/content/favorites.csv")
        return favs
    except FileNotFoundError:
        return "No favorites yet."



# Ranking System

In [ ]:
def clarity_score(text):
    words = text.split()
    avg_len = sum(len(w) for w in words) / (len(words)+1)
    return round(max(0, 100 - avg_len*5), 1)

def empathy_score(text):
    markers = ["sorry", "apologize", "thank", "understand", "appreciate", "क्षमा", "धन्यवाद"]
    score = sum(1 for m in markers if m.lower() in text.lower()) * 10
    return min(100, score+50)

def rank_excuses():
    df = pd.read_csv(HISTORY_FILE)
    df["rank"] = (df["clarity_score"] + df["empathy_score"]) / 2
    return df.sort_values(by="rank", ascending=False).head(5)

# Apology Mode (Guilt-tripping)

In [ ]:
def apology_excuse(scenario="family", urgency="medium", language="en"):
    prompt = (
        f"Write a heartfelt apology excuse.\n"
        f"Scenario: {scenario}\n"
        f"Urgency: {urgency}\n"
        f"Make it emotional and guilt-tripping."
    )
    output = excuse_generator(
        prompt,
        do_sample=True,
        max_new_tokens=80,
        pad_token_id=tokenizer.eos_token_id
    )
    excuse = output[0]["generated_text"].strip()
    if language == "hi":
        excuse = translate_to_hindi(excuse)
    return excuse

# 7. SMS/Email Drafts

In [ ]:
def sms_template(excuse):
    return f"SMS Draft:\nHey, just wanted to say {excuse}"

def email_template(excuse):
    return f"Subject: Quick Update\n\nDear Recipient,\n\n{excuse}\n\nThanks,\n[Your Name]"

#Voice Output

In [ ]:
def excuse_to_speech(text, filename="excuse.mp3"):
    tts = gTTS(text)
    tts.save(filename)
    return ipd.Audio(filename)

# Time-based Suggestions

In [ ]:
def time_based_suggestion():
    hour = datetime.now().hour
    if hour < 9: return "Morning: Suggest excuse about being late."
    elif hour < 18: return "Daytime: Suggest excuse about work/school tasks."
    else: return "Evening: Suggest excuse about missing social/family events."

#  Demo Usage

In [ ]:
print("🔹 Example Excuse (Work, Professional, Urgent, English):")
excuse, proof = generate_excuse("work", "professional", "high", "en")
print("Excuse:", excuse)
print("Supportive:", proof)
print(sms_template(excuse))
print(email_template(excuse))

print("\n🔹 Hindi Excuse (School, Sincere, Low):")
excuse, proof = generate_excuse("school", "sincere", "low", "hi")
print("Excuse:", excuse)

print("\n🔹 Apology Mode:")
print(apology_excuse("family", "high", "en"))

print("\n🔹 Time-based Suggestion:")
print(time_based_suggestion())

print("\n🔹 Top Ranked Excuses:")
print(rank_excuses())

print("\n🔹 History (last 3):")
print(show_history(3))

print("\n🔹 Converting excuse to speech (playable below):")
excuse_to_speech(excuse)

🔹 Example Excuse (Work, Professional, Urgent, English):
Excuse: Generate a short, believable excuse.
Scenario: work
Tone: professional
Urgency: high
Provide the excuse only, no extra text.

example: "I'm sorry to inform you that my father has passed away unexpectedly and I'll be unable to make it into work tomorrow. Please let my team know that I'll be working remotely for the rest of the week."

Your excuse: _______________________
Supportive: I’ll catch up during the weekend.
SMS Draft:
Hey, just wanted to say Generate a short, believable excuse.
Scenario: work
Tone: professional
Urgency: high
Provide the excuse only, no extra text.

example: "I'm sorry to inform you that my father has passed away unexpectedly and I'll be unable to make it into work tomorrow. Please let my team know that I'll be working remotely for the rest of the week."

Your excuse: _______________________
Subject: Quick Update

Dear Recipient,

Generate a short, believable excuse.
Scenario: work
Tone: professiona